In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("/content/PreviousDataset.csv")

In [3]:
numeric_cols = ['SSTA', 'ClimSST', 'Depth', 'Longitude_Degrees',
                'Latitude_Degrees', 'Average_Bleaching', 'Windspeed',
                'TSA', 'TSA_DHW', 'SSTA_DHW', 'SSTA_Minimum']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year

# Filter
df = df[df['SSTA'] > 0]
df = df[df['Year'] >= 2003]

In [4]:
df = df.sort_values(['Latitude_Degrees', 'Longitude_Degrees', 'Date'])
df['SSTA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['SSTA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['TSA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['TSA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

drop_cols = ['City_Town', 'City_Town_2', 'City_Town_3', 'Ocean', 'Realm', 'Ecoregion', 'ID', 'Country_Name', 'State_Island_Province', 'Date2', 'SSTA_Mean']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Month encoding
df['Month'] = df['Date'].dt.month
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df = df.drop(columns=['Month', 'Date'])

# Train

In [5]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [7]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
print(X_train_raw.isna().any().any())
print(X_test_raw.isna().any().any())

True
True


In [9]:
# Impute
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)


In [10]:
scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [11]:
pca = PCA(n_components=0.95) # Increased variance capture
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

## OpTuna

In [16]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_regression,
                                 k=trial.suggest_int("k_features", 5, X.shape[1]))),
        ('pca', PCA(
            n_components=trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-03 18:14:36,156] A new study created in memory with name: no-name-ad1d42b5-5058-4b96-b964-f202631abdfd
[I 2026-05-03 18:14:49,821] Trial 0 finished with value: 0.24095565083243584 and parameters: {'n_estimators': 175, 'max_depth': 23, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'k_features': 37, 'pca_variance': 0.9151093098183333}. Best is trial 0 with value: 0.24095565083243584.
[I 2026-05-03 18:14:52,352] Trial 1 finished with value: 0.1311489452328283 and parameters: {'n_estimators': 63, 'max_depth': 36, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'k_features': 5, 'pca_variance': 0.9412582396174853}. Best is trial 0 with value: 0.24095565083243584.
[I 2026-05-03 18:15:05,888] Trial 2 finished with value: 0.20179665140429895 and parameters: {'n_estimators': 280, 'max_depth': 25, 'min_samples_leaf': 5, 'max_features': 'log2', 'k_features': 8, 'pca_variance': 0.9245463971294889}. Best is trial 0 with value: 0.24095565083243584.
[I 2026-05-03 18:15:15,769] Trial 3 finis

========== RF BEST ==========
{'n_estimators': 212, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'k_features': 39, 'pca_variance': 0.9513585672920561}
Best R²: 0.2757


In [ ]:
def xgb_objective(trial):

    model = xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(
            score_func=f_regression,
            k=trial.suggest_int("k_features", 5, X.shape[1])
        )),
        ('pca', PCA(
            n_components=trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")

xgb_study.optimize(xgb_objective, n_trials=50)

print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-03 18:24:12,452] A new study created in memory with name: no-name-8609a070-b233-45b3-98d1-80507b22bf86
[I 2026-05-03 18:24:27,666] Trial 0 finished with value: 0.25513589511625157 and parameters: {'n_estimators': 191, 'learning_rate': 0.008692223883253747, 'max_depth': 13, 'subsample': 0.7335282891789726, 'colsample_bytree': 0.9585937099103312, 'k_features': 12, 'pca_variance': 0.9711057043696774}. Best is trial 0 with value: 0.25513589511625157.
[I 2026-05-03 18:24:30,998] Trial 1 finished with value: 0.28300871626827784 and parameters: {'n_estimators': 121, 'learning_rate': 0.08144715174478179, 'max_depth': 7, 'subsample': 0.882704348552916, 'colsample_bytree': 0.9438833355164007, 'k_features': 20, 'pca_variance': 0.9169597964989088}. Best is trial 1 with value: 0.28300871626827784.
[I 2026-05-03 18:24:40,429] Trial 2 finished with value: 0.2751939332038521 and parameters: {'n_estimators': 330, 'learning_rate': 0.06911685042454795, 'max_depth': 8, 'subsample': 0.5169705385

========== XGB BEST ==========
{'n_estimators': 344, 'learning_rate': 0.04803963260530461, 'max_depth': 6, 'subsample': 0.5878770997223028, 'colsample_bytree': 0.6678502421759117, 'k_features': 37, 'pca_variance': 0.9645180378308249}
Best R²: 0.3197


## Start!

In [19]:
xgb_model = xgb.XGBRegressor(
    n_estimators=316,
    learning_rate=0.02,
    max_depth=9,
    subsample=0.7,
    colsample_bytree=0.9,
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=131,
    max_depth=28,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42
)

In [20]:
voting = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.4556
RMSE: 7.1467


In [21]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# %%
# K-Fold Validation

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Train
    voting      = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
    voting.fit(X_train_pca, y_train)

    # Evaluate
    preds = voting.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.4087  |  RMSE: 7.2073
Fold 2  →  R²: 0.3857  |  RMSE: 8.5236
Fold 3  →  R²: 0.2067  |  RMSE: 8.6735
Fold 4  →  R²: 0.3774  |  RMSE: 6.5932

--- 4-Fold Summary ---
R²   mean: 0.3446  ±  0.0805
RMSE mean: 7.7494  ±  0.8781


## XGB + RF เข้า NN

ทำ stacking คล้าย voting แต่แทนที่จะเฉลี่ย prediction ของ XGB/RF จะเอา prediction ทั้งสองตัวเข้า Neural Network (`MLPRegressor`) เป็น meta-model


In [29]:
from sklearn.neural_network import MLPRegressor
from sklearn.base import clone

NN_RANDOM_STATE = 42


In [30]:
# 80/20 NN stacking: XGB/RF predictions -> NN meta-model

xgb_base_nn = clone(xgb_model)
rf_base_nn = clone(rf_model)

xgb_base_nn.fit(X_train_pca, y_train)
rf_base_nn.fit(X_train_pca, y_train)

meta_train_nn = np.column_stack([
    xgb_base_nn.predict(X_train_pca),
    rf_base_nn.predict(X_train_pca)
])
meta_test_nn = np.column_stack([
    xgb_base_nn.predict(X_test_pca),
    rf_base_nn.predict(X_test_pca)
])

nn_meta_model = MLPRegressor(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=2000,
    random_state=NN_RANDOM_STATE,
    early_stopping=True
)

nn_meta_model.fit(meta_train_nn, y_train)

nn_preds = nn_meta_model.predict(meta_test_nn)
nn_r2 = r2_score(y_test, nn_preds)
nn_rmse = np.sqrt(mean_squared_error(y_test, nn_preds))

print("========== XGB + RF -> NN STACKING (80/20 split) ==========")
print(f"NN stacking R²:   {nn_r2:.4f}")
print(f"NN stacking RMSE: {nn_rmse:.4f}")

if 'rmse' in globals() and 'r2' in globals():
    print()
    print("========== COMPARE WITH VOTING ==========")
    print(f"Voting R²:   {r2:.4f}")
    print(f"Voting RMSE: {rmse:.4f}")
    print(f"NN RMSE improvement over voting: {rmse - nn_rmse:.4f}")

    if nn_rmse < rmse:
        print("✅ NN stacking RMSE is better than voting.")
    else:
        print("❌ NN stacking RMSE is not better than voting.")


========== XGB + RF -> NN STACKING (80/20 split) ==========
NN stacking R²:   0.3684
NN stacking RMSE: 6.6405

========== COMPARE WITH VOTING ==========
Voting R²:   0.2438
Voting RMSE: 7.2665
NN RMSE improvement over voting: 0.6260
✅ NN stacking RMSE is better than voting.


In [31]:
# K-Fold Validation: XGB/RF predictions -> NN meta-model

n_splits = 4
kf_nn = KFold(n_splits=n_splits, shuffle=True, random_state=42)

nn_kfold_r2_scores = []
nn_kfold_rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf_nn.split(X), 1):
    X_train_raw_fold, X_test_raw_fold = X.iloc[train_idx], X.iloc[test_idx]
    y_train_fold, y_test_fold = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer_fold = SimpleImputer(strategy='median')
    X_train_imp_fold = imputer_fold.fit_transform(X_train_raw_fold)
    X_test_imp_fold = imputer_fold.transform(X_test_raw_fold)

    # Scale
    scaler_fold = StandardScaler()
    X_train_ss_fold = scaler_fold.fit_transform(X_train_imp_fold)
    X_test_ss_fold = scaler_fold.transform(X_test_imp_fold)

    # PCA same as original Start pipeline
    pca_fold = PCA(n_components=0.95)
    X_train_pca_fold = pca_fold.fit_transform(X_train_ss_fold)
    X_test_pca_fold = pca_fold.transform(X_test_ss_fold)

    # Base models from Start
    xgb_fold = clone(xgb_model)
    rf_fold = clone(rf_model)
    xgb_fold.fit(X_train_pca_fold, y_train_fold)
    rf_fold.fit(X_train_pca_fold, y_train_fold)

    # Meta features = base model predictions
    meta_train_fold = np.column_stack([
        xgb_fold.predict(X_train_pca_fold),
        rf_fold.predict(X_train_pca_fold)
    ])
    meta_test_fold = np.column_stack([
        xgb_fold.predict(X_test_pca_fold),
        rf_fold.predict(X_test_pca_fold)
    ])

    nn_fold = MLPRegressor(
        hidden_layer_sizes=(16, 8),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=NN_RANDOM_STATE,
        early_stopping=True
    )
    nn_fold.fit(meta_train_fold, y_train_fold)

    fold_preds = nn_fold.predict(meta_test_fold)
    fold_r2 = r2_score(y_test_fold, fold_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_test_fold, fold_preds))

    nn_kfold_r2_scores.append(fold_r2)
    nn_kfold_rmse_scores.append(fold_rmse)

    print(f"Fold {fold} → NN stacking R²: {fold_r2:.4f} | RMSE: {fold_rmse:.4f}")

print()
print(f"--- {n_splits}-Fold XGB + RF -> NN Summary ---")
print(f"R²   mean: {np.mean(nn_kfold_r2_scores):.4f} ± {np.std(nn_kfold_r2_scores):.4f}")
print(f"RMSE mean: {np.mean(nn_kfold_rmse_scores):.4f} ± {np.std(nn_kfold_rmse_scores):.4f}")


Fold 1 → NN stacking R²: 0.4031 | RMSE: 7.2411
Fold 2 → NN stacking R²: 0.4218 | RMSE: 8.2695
Fold 3 → NN stacking R²: 0.1862 | RMSE: 8.7847
Fold 4 → NN stacking R²: 0.3684 | RMSE: 6.6405

--- 4-Fold XGB + RF -> NN Summary ---
R²   mean: 0.3449 ± 0.0936
RMSE mean: 7.7340 ± 0.8410


## Mean Baseline

ใช้ค่าเฉลี่ยของ `Average_Bleaching` จาก train set เป็นคำตอบทุกแถว เพื่อเป็น naive baseline ว่าโมเดลของเราทำ RMSE ได้ดีกว่าการเดาค่าเฉลี่ยหรือไม่


In [27]:
# Mean baseline: predict every test row with train target mean
mean_baseline_value = y_train.mean()
mean_baseline_preds = np.full_like(y_test, mean_baseline_value, dtype=float)

mean_baseline_r2 = r2_score(y_test, mean_baseline_preds)
mean_baseline_rmse = np.sqrt(mean_squared_error(y_test, mean_baseline_preds))

print("========== MEAN BASELINE (80/20 split) ==========")
print(f"Mean Average_Bleaching from train set: {mean_baseline_value:.4f}")
print(f"Mean baseline R²:   {mean_baseline_r2:.4f}")
print(f"Mean baseline RMSE: {mean_baseline_rmse:.4f}")

if 'rmse' in globals() and 'r2' in globals():
    print()
    print("========== COMPARE WITH OUR ENSEMBLE ==========")
    print(f"Our ensemble R²:   {r2:.4f}")
    print(f"Our ensemble RMSE: {rmse:.4f}")
    print(f"RMSE improvement over mean baseline: {mean_baseline_rmse - rmse:.4f}")

    if rmse < mean_baseline_rmse:
        print("✅ Our ensemble RMSE is better than mean baseline.")
    else:
        print("❌ Our ensemble RMSE is not better than mean baseline.")
else:
    print("Run the ensemble evaluation cell first to compare with our model.")


========== MEAN BASELINE (80/20 split) ==========
Mean Average_Bleaching from train set: 2.9582
Mean baseline R²:   -0.0061
Mean baseline RMSE: 8.3813

========== COMPARE WITH OUR ENSEMBLE ==========
Our ensemble R²:   0.2438
Our ensemble RMSE: 7.2665
RMSE improvement over mean baseline: 1.1148
✅ Our ensemble RMSE is better than mean baseline.


In [28]:
# K-Fold Mean Baseline: each fold predicts with that fold's train target mean
n_splits = 4
kf_mean_baseline = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mean_baseline_kfold_results = []

for fold, (train_idx, test_idx) in enumerate(kf_mean_baseline.split(X), 1):
    y_fold_train = Y.iloc[train_idx]
    y_fold_test = Y.iloc[test_idx]

    fold_mean = y_fold_train.mean()
    fold_preds = np.full_like(y_fold_test, fold_mean, dtype=float)

    fold_r2 = r2_score(y_fold_test, fold_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_fold_test, fold_preds))

    mean_baseline_kfold_results.append({
        'Fold': fold,
        'Train_Mean_Average_Bleaching': fold_mean,
        'R²': fold_r2,
        'RMSE': fold_rmse,
    })

    print(f"Fold {fold} → mean: {fold_mean:.4f} | R²: {fold_r2:.4f} | RMSE: {fold_rmse:.4f}")

mean_baseline_kfold_df = pd.DataFrame(mean_baseline_kfold_results)

print()
print(f"--- {n_splits}-Fold Mean Baseline Summary ---")
print(f"R²   mean: {mean_baseline_kfold_df['R²'].mean():.4f} ± {mean_baseline_kfold_df['R²'].std():.4f}")
print(f"RMSE mean: {mean_baseline_kfold_df['RMSE'].mean():.4f} ± {mean_baseline_kfold_df['RMSE'].std():.4f}")

display(mean_baseline_kfold_df)


Fold 1 → mean: 2.8285 | R²: -0.0002 | RMSE: 9.3734
Fold 2 → mean: 2.6633 | R²: -0.0024 | RMSE: 10.8880
Fold 3 → mean: 2.7309 | R²: -0.0007 | RMSE: 9.7413
Fold 4 → mean: 2.9582 | R²: -0.0061 | RMSE: 8.3813

--- 4-Fold Mean Baseline Summary ---
R²   mean: -0.0023 ± 0.0027
RMSE mean: 9.5960 ± 1.0353


,Fold,Train_Mean_Average_Bleaching,R²,RMSE
0,1,2.828460,-0.000201,9.373428
1,2,2.663343,-0.002353,10.887962
2,3,2.730890,-0.000698,9.741314
3,4,2.958185,-0.006086,8.381296


## ลอง 5pcs


In [22]:
import optuna
from sklearn.model_selection import KFold
from sklearn.ensemble import VotingRegressor
from sklearn.base import clone

# Fixed PCA setup for this experiment
N_COMPONENTS_5PCS = 5
N_TRIALS_5PCS = 50

cv_5pcs = KFold(n_splits=4, shuffle=True, random_state=42)


In [23]:
# Optuna: Random Forest with fixed 5 PCs

def rf_5pcs_objective(trial):
    fold_scores = []

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 40),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'n_jobs': -1,
        'random_state': 42,
    }

    for train_idx, valid_idx in cv_5pcs.split(X_train_raw):
        X_fold_train_raw = X_train_raw.iloc[train_idx]
        X_fold_valid_raw = X_train_raw.iloc[valid_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        imputer_5pcs = SimpleImputer(strategy='median')
        X_fold_train_imp = imputer_5pcs.fit_transform(X_fold_train_raw)
        X_fold_valid_imp = imputer_5pcs.transform(X_fold_valid_raw)

        scaler_5pcs = StandardScaler()
        X_fold_train_ss = scaler_5pcs.fit_transform(X_fold_train_imp)
        X_fold_valid_ss = scaler_5pcs.transform(X_fold_valid_imp)

        pca_5pcs = PCA(n_components=N_COMPONENTS_5PCS)
        X_fold_train_pca = pca_5pcs.fit_transform(X_fold_train_ss)
        X_fold_valid_pca = pca_5pcs.transform(X_fold_valid_ss)

        model = RandomForestRegressor(**params)
        model.fit(X_fold_train_pca, y_fold_train)
        preds = model.predict(X_fold_valid_pca)
        fold_scores.append(r2_score(y_fold_valid, preds))

    return np.mean(fold_scores)

rf_5pcs_study = optuna.create_study(direction='maximize')
rf_5pcs_study.optimize(rf_5pcs_objective, n_trials=N_TRIALS_5PCS)

print('========== RF 5PCS BEST ==========')
print(rf_5pcs_study.best_params)
print(f"Best CV R²: {rf_5pcs_study.best_value:.4f}")


[I 2026-05-03 18:32:32,143] A new study created in memory with name: no-name-79e794ca-fb31-4056-92a1-cc80acc95cca
[I 2026-05-03 18:32:37,024] Trial 0 finished with value: 0.12123697334641206 and parameters: {'n_estimators': 168, 'max_depth': 13, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.12123697334641206.
[I 2026-05-03 18:32:46,147] Trial 1 finished with value: 0.14957796166468254 and parameters: {'n_estimators': 223, 'max_depth': 26, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.14957796166468254.
[I 2026-05-03 18:32:47,674] Trial 2 finished with value: 0.09639131766835007 and parameters: {'n_estimators': 53, 'max_depth': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.14957796166468254.
[I 2026-05-03 18:32:54,063] Trial 3 finished with value: 0.10865522082317303 and parameters: {'n_estimators': 299, 'max_depth': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 1 with v

========== RF 5PCS BEST ==========
{'n_estimators': 170, 'max_depth': 27, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best CV R²: 0.1520


In [24]:
# Optuna: XGBoost with fixed 5 PCs

def xgb_5pcs_objective(trial):
    fold_scores = []

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'n_jobs': -1,
        'random_state': 42,
    }

    for train_idx, valid_idx in cv_5pcs.split(X_train_raw):
        X_fold_train_raw = X_train_raw.iloc[train_idx]
        X_fold_valid_raw = X_train_raw.iloc[valid_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        imputer_5pcs = SimpleImputer(strategy='median')
        X_fold_train_imp = imputer_5pcs.fit_transform(X_fold_train_raw)
        X_fold_valid_imp = imputer_5pcs.transform(X_fold_valid_raw)

        scaler_5pcs = StandardScaler()
        X_fold_train_ss = scaler_5pcs.fit_transform(X_fold_train_imp)
        X_fold_valid_ss = scaler_5pcs.transform(X_fold_valid_imp)

        pca_5pcs = PCA(n_components=N_COMPONENTS_5PCS)
        X_fold_train_pca = pca_5pcs.fit_transform(X_fold_train_ss)
        X_fold_valid_pca = pca_5pcs.transform(X_fold_valid_ss)

        model = xgb.XGBRegressor(**params)
        model.fit(X_fold_train_pca, y_fold_train)
        preds = model.predict(X_fold_valid_pca)
        fold_scores.append(r2_score(y_fold_valid, preds))

    return np.mean(fold_scores)

xgb_5pcs_study = optuna.create_study(direction='maximize')
xgb_5pcs_study.optimize(xgb_5pcs_objective, n_trials=N_TRIALS_5PCS)

print('========== XGB 5PCS BEST ==========')
print(xgb_5pcs_study.best_params)
print(f"Best CV R²: {xgb_5pcs_study.best_value:.4f}")


[I 2026-05-03 18:38:54,502] A new study created in memory with name: no-name-7e892995-c32c-496c-9de1-e8865158c43b
[I 2026-05-03 18:39:03,523] Trial 0 finished with value: 0.11464382090643885 and parameters: {'n_estimators': 241, 'learning_rate': 0.09530707040158423, 'max_depth': 12, 'subsample': 0.7131544463246029, 'colsample_bytree': 0.9901771861011701}. Best is trial 0 with value: 0.11464382090643885.
[I 2026-05-03 18:39:04,768] Trial 1 finished with value: 0.11127242335011239 and parameters: {'n_estimators': 210, 'learning_rate': 0.02802362137989371, 'max_depth': 5, 'subsample': 0.6841865981286752, 'colsample_bytree': 0.8387341877410801}. Best is trial 0 with value: 0.11464382090643885.
[I 2026-05-03 18:39:06,974] Trial 2 finished with value: 0.10540867053004718 and parameters: {'n_estimators': 115, 'learning_rate': 0.041142869753984, 'max_depth': 10, 'subsample': 0.9309986102229684, 'colsample_bytree': 0.6911968125228601}. Best is trial 0 with value: 0.11464382090643885.
[I 2026-05

========== XGB 5PCS BEST ==========
{'n_estimators': 289, 'learning_rate': 0.0088685316344283, 'max_depth': 12, 'subsample': 0.8692460481580794, 'colsample_bytree': 0.8108971761996141}
Best CV R²: 0.1575


In [25]:
# Train/Test evaluation: RF, XGB, Ensemble with fixed 5 PCs

imputer_5pcs = SimpleImputer(strategy='median')
X_train_imp_5pcs = imputer_5pcs.fit_transform(X_train_raw)
X_test_imp_5pcs = imputer_5pcs.transform(X_test_raw)

scaler_5pcs = StandardScaler()
X_train_ss_5pcs = scaler_5pcs.fit_transform(X_train_imp_5pcs)
X_test_ss_5pcs = scaler_5pcs.transform(X_test_imp_5pcs)

pca_5pcs = PCA(n_components=N_COMPONENTS_5PCS)
X_train_pca_5pcs = pca_5pcs.fit_transform(X_train_ss_5pcs)
X_test_pca_5pcs = pca_5pcs.transform(X_test_ss_5pcs)

best_rf_5pcs = RandomForestRegressor(
    **rf_5pcs_study.best_params,
    n_jobs=-1,
    random_state=42
)

best_xgb_5pcs = xgb.XGBRegressor(
    **xgb_5pcs_study.best_params,
    objective='reg:squarederror',
    eval_metric='rmse',
    n_jobs=-1,
    random_state=42
)

ensemble_5pcs = VotingRegressor([
    ('xgb', best_xgb_5pcs),
    ('rf', best_rf_5pcs)
])

models_5pcs = {
    'RF': best_rf_5pcs,
    'XGB': best_xgb_5pcs,
    'Ensemble': ensemble_5pcs,
}

results_5pcs = []
for model_name, model in models_5pcs.items():
    model.fit(X_train_pca_5pcs, y_train)
    preds = model.predict(X_test_pca_5pcs)
    results_5pcs.append({
        'Model': model_name,
        'R²': r2_score(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
    })

results_5pcs_df = pd.DataFrame(results_5pcs)
print('========== 5PCS TRAIN/TEST RESULTS ==========')
print(results_5pcs_df.to_string(index=False))


========== 5PCS TRAIN/TEST RESULTS ==========
   Model       R²     RMSE
      RF 0.244380 7.263489
     XGB 0.221804 7.371199
Ensemble 0.243756 7.266490


In [26]:
# K-Fold Validation: RF, XGB, Ensemble with fixed 5 PCs

n_splits = 4
kf_5pcs = KFold(n_splits=n_splits, shuffle=True, random_state=42)

kfold_results_5pcs = []

for fold, (train_idx, test_idx) in enumerate(kf_5pcs.split(X), 1):
    X_fold_train_raw = X.iloc[train_idx]
    X_fold_test_raw = X.iloc[test_idx]
    y_fold_train = Y.iloc[train_idx]
    y_fold_test = Y.iloc[test_idx]

    imputer_fold = SimpleImputer(strategy='median')
    X_fold_train_imp = imputer_fold.fit_transform(X_fold_train_raw)
    X_fold_test_imp = imputer_fold.transform(X_fold_test_raw)

    scaler_fold = StandardScaler()
    X_fold_train_ss = scaler_fold.fit_transform(X_fold_train_imp)
    X_fold_test_ss = scaler_fold.transform(X_fold_test_imp)

    pca_fold = PCA(n_components=N_COMPONENTS_5PCS)
    X_fold_train_pca = pca_fold.fit_transform(X_fold_train_ss)
    X_fold_test_pca = pca_fold.transform(X_fold_test_ss)

    fold_models = {
        'RF': RandomForestRegressor(
            **rf_5pcs_study.best_params,
            n_jobs=-1,
            random_state=42
        ),
        'XGB': xgb.XGBRegressor(
            **xgb_5pcs_study.best_params,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=-1,
            random_state=42
        ),
    }
    fold_models['Ensemble'] = VotingRegressor([
        ('xgb', clone(fold_models['XGB'])),
        ('rf', clone(fold_models['RF']))
    ])

    for model_name, model in fold_models.items():
        model.fit(X_fold_train_pca, y_fold_train)
        preds = model.predict(X_fold_test_pca)
        r2 = r2_score(y_fold_test, preds)
        rmse = np.sqrt(mean_squared_error(y_fold_test, preds))

        kfold_results_5pcs.append({
            'Fold': fold,
            'Model': model_name,
            'R²': r2,
            'RMSE': rmse,
        })
        print(f"Fold {fold} | {model_name:8s} → R²: {r2:.4f} | RMSE: {rmse:.4f}")

kfold_results_5pcs_df = pd.DataFrame(kfold_results_5pcs)

print()
print(f"--- {n_splits}-Fold 5PCS Summary ---")
summary_5pcs = kfold_results_5pcs_df.groupby('Model').agg({
    'R²': ['mean', 'std'],
    'RMSE': ['mean', 'std']
})
print(summary_5pcs)


Fold 1 | RF       → R²: 0.2552 | RMSE: 8.0887
Fold 1 | XGB      → R²: 0.2143 | RMSE: 8.3079
Fold 1 | Ensemble → R²: 0.2422 | RMSE: 8.1590
Fold 2 | RF       → R²: 0.2190 | RMSE: 9.6111
Fold 2 | XGB      → R²: 0.2120 | RMSE: 9.6536
Fold 2 | Ensemble → R²: 0.2196 | RMSE: 9.6072
Fold 3 | RF       → R²: 0.0961 | RMSE: 9.2583
Fold 3 | XGB      → R²: 0.1216 | RMSE: 9.1268
Fold 3 | Ensemble → R²: 0.1172 | RMSE: 9.1497
Fold 4 | RF       → R²: 0.2444 | RMSE: 7.2635
Fold 4 | XGB      → R²: 0.2218 | RMSE: 7.3712
Fold 4 | Ensemble → R²: 0.2438 | RMSE: 7.2665

--- 4-Fold 5PCS Summary ---
                R²                RMSE          
              mean       std      mean       std
Model                                           
Ensemble  0.205674  0.060031  8.545594  1.045241
RF        0.203651  0.073300  8.555396  1.079404
XGB       0.192419  0.047420  8.614889  0.997008
